In [3]:
import pandas as pd
import numpy as np
import json
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

# 1. Load Data
df = pd.read_csv("/data/demo_data/football_test/team_stats.csv")
with open("/data/demo_data/football_test/match_api_metric_map.json", 'r') as f:
    metric_map = json.load(f)

# 2. Preprocessing
cols_to_exclude = ['team_id', 'club_name', 'competition_id', 'season']
features = df.drop(columns=cols_to_exclude).select_dtypes(include=[np.number])
features = features.fillna(features.mean())

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# 3. Determine Optimal Factors using Reduced Correlation Matrix (SMC)
corr_matrix = np.corrcoef(X_scaled, rowvar=False)
# Squared Multiple Correlations as communality estimates
inv_corr = np.linalg.inv(corr_matrix)
smc = 1 - 1 / np.diag(inv_corr)

# Replace diagonal with communalities
reduced_corr = corr_matrix.copy()
np.fill_diagonal(reduced_corr, smc)
fa_eigenvalues = sorted(np.linalg.eigvalsh(reduced_corr), reverse=True)

# Determine number of factors where eigenvalues of reduced matrix > 1
n_factors = sum(ev > 1 for ev in fa_eigenvalues)

# 4. Perform Factor Analysis (Maximum Likelihood/Latent Variable)
fa = FactorAnalysis(n_components=n_factors, rotation='varimax', random_state=42)
fa.fit(X_scaled)

# 5. Extract and Name Loadings


feature_names = [metric_map.get(col, col) for col in features.columns]
loadings_df = pd.DataFrame(
    fa.components_.T, 
    index=feature_names, 
    columns=[f"Factor_{i+1}" for i in range(n_factors)]
)

# Export the detailed loading matrix
loadings_df.to_csv('factor_analysis_loadings.csv')

In [4]:
loadings_df

,Factor_1,Factor_2,Factor_3,Factor_4,Factor_5,Factor_6,Factor_7,Factor_8,Factor_9,Factor_10,Factor_11,Factor_12,Factor_13
Fouls committed,-0.537628,0.069097,0.003240,0.024283,-0.012072,-0.287850,-0.035202,-0.272470,0.329892,-0.001617,-0.064781,-0.210600,0.171493
num_throwins_final_third,-0.134562,0.046636,-0.012448,0.041340,0.036869,-0.291799,0.083335,0.123089,0.448227,0.086508,0.224234,0.028281,0.468862
Offsides,0.017663,-0.082343,-0.098638,0.046706,0.163445,-0.117846,0.075263,0.014092,0.112684,-0.011140,-0.020393,-0.205868,0.115684
Shots on target,0.904328,-0.043827,0.051155,0.084645,-0.148736,-0.214463,0.101183,0.005106,0.100052,-0.037083,-0.107610,-0.044535,-0.013260
Shots,0.890909,0.028103,-0.117510,0.011875,-0.291246,-0.258125,0.077438,-0.043392,0.153482,-0.011480,-0.069996,0.022164,0.055950
...,...,...,...,...,...,...,...,...,...,...,...,...,...
opp_shots_from_outside_box_pct,-0.113757,-0.045843,-0.004200,0.047742,-0.096471,-0.420393,-0.037444,-0.096931,-0.030754,-0.548946,0.021057,0.015931,0.044141
opp_shots_per_final_third_pass,0.056165,-0.036725,0.041363,0.109802,-0.153121,-0.081576,-0.063405,0.075617,0.669466,0.026445,-0.002416,-0.224366,-0.098464
opp_shots_from_direct_attacks_pct,0.360900,0.068332,-0.046122,0.072315,-0.045095,-0.175043,0.038708,0.171407,0.526539,0.084838,0.058230,-0.050848,-0.110656
opp_shots_from_sustained_attacks_pct,-0.265395,-0.003061,-0.031961,-0.067104,0.087150,0.267504,-0.118692,-0.014611,-0.629458,-0.047735,0.076285,0.060053,0.099241


1. Factor 1: Offensive Threat & Chance Creation
Primary Metrics: np xG, xG, High opportunity shots, Shots on target.

Description: This factor represents a team's fundamental ability to create high-quality scoring opportunities through open play and sustained pressure.

2. Factor 2: Defensive Fragility & Match Underperformance
Primary Metrics: Opp. np Goals, Opp. Goals, Red cards (Positive); Points, Goal Difference (Negative).

Description: High scores here indicate a team that concedes frequently, suffers from discipline issues (red cards), and tends to lose matches/points.

3. Factor 3: Finishing Clinicality
Primary Metrics: np Goals, Goals.

Description: This captures the "pure finishing" aspect—converting the chances created into actual goals, independent of the volume of shots.

4. Factor 4: Penalty & Set-Piece Reliance
Primary Metrics: Penalties, Opp. Red Cards.

Description: Highlights teams that generate a significant portion of their danger or goals through penalties and drawing fouls from opponents.

5. Factor 5: Shot Quality & Selection Efficiency
Primary Metrics: np xG per shot (Positive); Shots from outside the box % (Negative).

Description: Distinguishes teams that are disciplined in their shot selection, preferring high-probability shots close to the goal over speculative long-distance efforts.

6. Factor 6: Opponent Dominance & Defensive Load
Primary Metrics: Opp. np xG, Opp. xG, Opp. Box touches.

Description: Measures how much the team is "pinned back." High values indicate the opponent is spending a lot of time in the team's box and creating chances.

7. Factor 7: Immediate Recovery & Possession Stability
Primary Metrics: Recoveries, Possessions retained after 5s, xT within 10s after recovery.

Description: Focuses on the "transition to defense." It measures the ability to win the ball back and immediately establish controlled possession or a counter-threat.

8. Factor 8: Patient Build-up & Methodical Attack
Primary Metrics: Shots from sustained attacks %, Possessions retained after 5s % (Positive); Box to shot % (Negative).

Description: Represents a "possession-heavy" style where the team cycles the ball in the final third and waits for a specific opening rather than shooting early.

9. Factor 9: Aggressive High-Pressing
Primary Metrics: Defensive intensity (Positive); PPDA, Time to defensive action (Negative).

Description: Measures the intensity of the team's press. Low PPDA and low time-to-action combined with high intensity indicate an aggressive "Gegenpressing" style.

10. Factor 10: Opponent Shot Quality Suppression
Primary Metrics: Opp. np xG per shot (Positive); Opp. shots from outside box % (Negative).

Description: This factor reflects defensive organization that successfully forces the opponent into taking low-quality, long-distance shots.

11. Factor 11: Wing Play & Cross-Heavy Attack
Primary Metrics: Crosses per final third possession, Box entries from crosses, Long ball %.

Description: Captures a tactical reliance on width and aerial service into the box as a primary means of entry.

12. Factor 12: Defensive Discipline & Low-Block Dynamics
Primary Metrics: Median time to 1st forward pass, Opp. penalties (Negative).

Description: Relates to the speed of clearing the ball from the defensive third and the discipline (avoiding penalties) shown when defending deep.

13. Factor 13: Verticality & Rapid Transitions
Primary Metrics: Final third entry within 10s after recovery, Forward passes from middle third %.

Description: Represents a direct, vertical playstyle that looks to exploit the opponent's transition phase as quickly as possible.

In [5]:
# Dictionary mapping Factor IDs to the names provided above
factor_name_map = {
    "Factor_1": "Offensive Threat & Chance Creation",
    "Factor_2": "Defensive Fragility & Match Underperformance",
    "Factor_3": "Finishing Clinicality",
    "Factor_4": "Penalty & Set-Piece Reliance",
    "Factor_5": "Shot Quality & Selection Efficiency",
    "Factor_6": "Opponent Dominance & Defensive Load",
    "Factor_7": "Immediate Recovery & Possession Stability",
    "Factor_8": "Patient Build-up & Methodical Attack",
    "Factor_9": "Aggressive High-Pressing",
    "Factor_10": "Opponent Shot Quality Suppression",
    "Factor_11": "Wing Play & Cross-Heavy Attack",
    "Factor_12": "Defensive Discipline & Low-Block Dynamics",
    "Factor_13": "Verticality & Rapid Transitions"
}

# Apply to your loadings DataFrame (assuming it was created in the previous step)
loadings_df.rename(columns=factor_name_map, inplace=True)

In [6]:
loadings_df

,Offensive Threat & Chance Creation,Defensive Fragility & Match Underperformance,Finishing Clinicality,Penalty & Set-Piece Reliance,Shot Quality & Selection Efficiency,Opponent Dominance & Defensive Load,Immediate Recovery & Possession Stability,Patient Build-up & Methodical Attack,Aggressive High-Pressing,Opponent Shot Quality Suppression,Wing Play & Cross-Heavy Attack,Defensive Discipline & Low-Block Dynamics,Verticality & Rapid Transitions
Fouls committed,-0.537628,0.069097,0.003240,0.024283,-0.012072,-0.287850,-0.035202,-0.272470,0.329892,-0.001617,-0.064781,-0.210600,0.171493
num_throwins_final_third,-0.134562,0.046636,-0.012448,0.041340,0.036869,-0.291799,0.083335,0.123089,0.448227,0.086508,0.224234,0.028281,0.468862
Offsides,0.017663,-0.082343,-0.098638,0.046706,0.163445,-0.117846,0.075263,0.014092,0.112684,-0.011140,-0.020393,-0.205868,0.115684
Shots on target,0.904328,-0.043827,0.051155,0.084645,-0.148736,-0.214463,0.101183,0.005106,0.100052,-0.037083,-0.107610,-0.044535,-0.013260
Shots,0.890909,0.028103,-0.117510,0.011875,-0.291246,-0.258125,0.077438,-0.043392,0.153482,-0.011480,-0.069996,0.022164,0.055950
...,...,...,...,...,...,...,...,...,...,...,...,...,...
opp_shots_from_outside_box_pct,-0.113757,-0.045843,-0.004200,0.047742,-0.096471,-0.420393,-0.037444,-0.096931,-0.030754,-0.548946,0.021057,0.015931,0.044141
opp_shots_per_final_third_pass,0.056165,-0.036725,0.041363,0.109802,-0.153121,-0.081576,-0.063405,0.075617,0.669466,0.026445,-0.002416,-0.224366,-0.098464
opp_shots_from_direct_attacks_pct,0.360900,0.068332,-0.046122,0.072315,-0.045095,-0.175043,0.038708,0.171407,0.526539,0.084838,0.058230,-0.050848,-0.110656
opp_shots_from_sustained_attacks_pct,-0.265395,-0.003061,-0.031961,-0.067104,0.087150,0.267504,-0.118692,-0.014611,-0.629458,-0.047735,0.076285,0.060053,0.099241


In [7]:
import pandas as pd
import numpy as np
import json
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Load Data
df = pd.read_csv("/data/demo_data/football_test/team_stats.csv")
with open("/data/demo_data/football_test/match_api_metric_map.json", 'r') as f:
    metric_map = json.load(f)


cols_to_exclude = ['team_id', 'club_name', 'competition_id', 'season']
features = df.drop(columns=cols_to_exclude).select_dtypes(include=[np.number]).fillna(df.mean(numeric_only=True))

# 2. Standardize and Apply Factor Analysis (13 Factors)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
fa = FactorAnalysis(n_components=13, rotation='varimax', random_state=42)
factor_scores = fa.fit_transform(X_scaled)

# 3. Cluster Analysis (K-Means)
# Using K=7 based on silhouette optimization
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(factor_scores)

# 4. Analyze Cluster Profiles
factor_names = [
    "Offensive Threat", "Defensive Fragility", "Finishing Clinicality", "Penalty Reliance",
    "Shot Quality", "Opponent Dominance", "Recovery & Retention", "Patient Build-up",
    "High Pressing", "Shot Suppression", "Wing Play", "Defensive Discipline", "Vertical Transitions"
]
factor_df = pd.DataFrame(factor_scores, columns=factor_names)
factor_df['cluster'] = df['cluster']
cluster_centroids = factor_df.groupby('cluster').mean()

# Output results
print(cluster_centroids)
#df.to_csv('team_clusters.csv', index=False)

         Offensive Threat  Defensive Fragility  Finishing Clinicality  \
cluster                                                                 
0               -0.720692            -0.067045              -0.095552   
1               -0.143812            -0.107408               0.369125   
2               -0.816061            -0.094322               0.315621   
3                0.689522             0.525956              -0.778868   
4               -0.334252            -0.041706              -0.023695   
5                0.826747            -0.090226              -0.072320   
6                1.728282            -0.202162               0.689624   

         Penalty Reliance  Shot Quality  Opponent Dominance  \
cluster                                                       
0                0.059034     -0.065078            0.499717   
1               -0.254452     -0.107738            0.696585   
2               -0.024496      0.683871           -0.808358   
3               -0.064725  

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Cluster 1: Defensive Wing-Back Resisters.
This cluster consists of teams that prioritize defensive organization and wide play while often conceding territorial control to the opponent. They are exceptionally effective at suppressing the quality of opponent shots and utilizing wing-based attacks to move the ball forward. Their primary struggle is a lack of overall offensive threat and a consistent inability to create high-quality scoring chances in the final third.

Cluster 2: Methodical Possession Conservers.
These teams prioritize ball retention and patient build-up, typically favoring play through the center of the pitch rather than the wings. They excel at winning the ball back and maintaining possession stability through methodical passing sequences and high recovery rates. However, their avoidance of wing play and slow vertical transitions often makes their attack predictable and easier for organized defenses to contain.

Cluster 3: High-Pressing Counter-Punchers.
This group utilizes an aggressive high press to disrupt opponents and looks for quick vertical transitions immediately after winning the ball. They are highly effective at pressing high up the pitch and suppressing opponent dominance through persistent defensive activity. Despite their high work rate, they struggle significantly to generate a high volume of offensive shots and consistent goal-scoring opportunities.

Cluster 4: Inefficient Dominators.
Teams in this cluster are characterized by their ability to create high-quality chances while simultaneously suffering from poor finishing and defensive lapses. They are skilled at generating high-quality shots and recovering the ball effectively to sustain pressure on the opponent. Their lack of clinical finishing and inherent defensive fragility often prevents them from converting their statistical dominance into actual match results.

Cluster 5: Disciplined Volume Shooters.
This cluster consists of teams that rely on a disciplined defensive structure but struggle with the efficiency of their shooting. They maintain a high level of defensive discipline and are successful at avoiding conceded penalties or defensive errors. Conversely, they suffer from poor shot selection—often settling for low-probability attempts—and an inability to suppress high-quality chances created by their opponents.

Cluster 6: Direct Vertical Attackers.
These teams focus on rapid, vertical transitions to exploit spaces through the middle of the pitch as quickly as possible. They possess a potent offensive threat and are elite at moving the ball vertically during the crucial transitional phases of the game. Unfortunately, they are extremely vulnerable once possession is lost due to very poor recovery rates and a lack of defensive stability after turnovers.

Cluster 7: Elite All-Rounders.
This group represents the top-tier performers who dominate the league both offensively and defensively across almost all metrics. They combine an elite offensive threat with clinical finishing and an effective high-pressing system that successfully suppresses opponent shot quality. Their only relative tactical drawback is a slower transition speed, as they often prefer to control the match through sustained pressure rather than rapid counter-attacks.

In [8]:
import pandas as pd
import numpy as np
import json
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans



# 2. Preprocessing
cols_to_exclude = ['team_id', 'club_name', 'competition_id', 'season']
features = df.drop(columns=cols_to_exclude).select_dtypes(include=[np.number]).fillna(df.mean(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# 3. Factor Analysis (13 Factors)
n_factors = 13
fa = FactorAnalysis(n_components=n_factors, rotation='varimax', random_state=42)
factor_scores = fa.fit_transform(X_scaled)

factor_names = [
    "Offensive Threat", "Defensive Fragility", "Finishing Clinicality", "Penalty Reliance",
    "Shot Quality", "Opponent Dominance", "Recovery & Retention", "Patient Build-up",
    "High Pressing", "Shot Suppression", "Wing Play", "Defensive Discipline", "Vertical Transitions"
]

# 4. Cluster Analysis (K=7)
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(factor_scores)

# 5. Extract Real Madrid Profile
df_analysis = df[['club_name', 'season']].copy()
for i, name in enumerate(factor_names):
    df_analysis[name] = factor_scores[:, i]
df_analysis['cluster'] = cluster_labels

real_madrid = df_analysis[df_analysis['club_name'].str.contains("Real Madrid", case=False)]

# Display Factor Scores for Real Madrid
print("Real Madrid Factor Profile:")
print(real_madrid[factor_names].T)
print(f"\nAssigned Cluster: {real_madrid['cluster'].values[0]}")

Real Madrid Factor Profile:
                             19
Offensive Threat       1.690995
Defensive Fragility    0.267053
Finishing Clinicality -1.724045
Penalty Reliance       2.248151
Shot Quality          -1.255797
Opponent Dominance    -0.712461
Recovery & Retention  -0.597380
Patient Build-up       0.195616
High Pressing         -1.318721
Shot Suppression       0.449292
Wing Play             -1.818957
Defensive Discipline   0.144647
Vertical Transitions  -0.115184

Assigned Cluster: 5


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
